# Source-supported posterior: finite theoretical witnesses

Proofs and assumptions: [theory_main.md](theory_main.md). These examples verify finite algebra and counterexamples. No learned HSE, LLapDiff training or real cross-dataset experiment is performed. Zero internal null coordinates are not public reconstruction claims.

In [ ]:
import csv, os
from pathlib import Path
import numpy as np
rows=[]
def record(name,value):
    rows.append(dict(witness=name,value=float(value),scope='finite_theory_only'))
    print(name, float(value))
I=np.eye(4)
# Two source designs: common axis0; private axes1/2; source-null axis3.
Oa=np.diag([1.,1.,0.,0.]); Ob=np.diag([1.,0.,1.,0.])
Us=np.diag([1.,1.,1.,0.]); Cs=np.diag([1.,0.,0.,0.])
blocks=[Cs,Oa-Cs,Us-Oa,I-Us]
np.testing.assert_allclose(sum(blocks),I)
for j,P in enumerate(blocks):
    np.testing.assert_allclose(P@P,P)
    for k,Q in enumerate(blocks):
        if j!=k: np.testing.assert_allclose(P@Q,0.)
record('source_partition_sum_residual',np.linalg.norm(sum(blocks)-I))
record('source_role_rank_each',min(np.trace(P) for P in blocks))
# Target sees source-private axes, but loses the common source axis.
Ot=np.diag([0.,1.,1.,0.]); Ct=np.zeros((4,4))
target_blocks=[Ct,Ot-Ct,Us-Ot,I-Us]
np.testing.assert_allclose(sum(target_blocks),I)
assert np.trace(Cs)==1 and np.trace(Ct)==0
assert np.trace(Us-Ot)==1
record('target_common_rank',np.trace(Ct))
record('target_missing_rank',np.trace(Us-Ot))

## Sensitivity is not an individually observable coordinate
The rank-one operator sees a sum. A diagonal-information test would incorrectly admit both coordinates as separately recovered.

In [ ]:
A=np.array([[1.,1.]])
a=np.array([2.,3.]); b=a+np.array([1.,-1.])
assert np.all(np.diag(A.T@A)>0)
np.testing.assert_allclose(A@a,A@b)
record('mixed_operator_ambiguous_state_distance',np.linalg.norm(a-b))
record('mixed_operator_observation_difference',np.linalg.norm(A@a-A@b))
# Two indistinguishable unpaired-data worlds, opposite missing conditionals.
joint_plus=np.eye(2)/2; joint_minus=np.fliplr(np.eye(2))/2
np.testing.assert_allclose(joint_plus.sum(0),joint_minus.sum(0))
np.testing.assert_allclose(joint_plus.sum(1),joint_minus.sum(1))
conditional_gap=abs(joint_plus[1,1]/.5-joint_minus[1,1]/.5)
record('unpaired_same_marginal_conditional_gap',conditional_gap)
assert conditional_gap==1

## Source-null is a likelihood boundary, not prior independence
Both cases use the identical observed operator and noise variance. Only the assumed prior correlation differs.

In [ ]:
rho=.8; prior=np.array([[1.,rho],[rho,1.]])
operator=np.array([[1.,0.]])
gain=prior@operator.T/(float((operator@prior@operator.T).item())+1)
mean=gain[:,0] # X=1
post=prior-gain@operator@prior
np.testing.assert_allclose(mean,[.5,.4])
np.testing.assert_allclose(post[1,1],.68)
record('correlated_prior_null_posterior_mean',mean[1])
record('correlated_prior_null_posterior_variance',post[1,1])
independent=np.eye(2); gain_i=independent@operator.T/2
post_i=independent-gain_i@operator@independent
record('independent_prior_null_posterior_variance',post_i[1,1])
assert post_i[1,1]==1

## Projection permits only the admitted update
One source-supported missing axis is admitted. Random proposals deliberately alter every coordinate; projection prevents the forbidden changes. This does not establish posterior correctness.

In [ ]:
G=np.diag([0.,0.,1.,0.]); observed=np.array([.4,-.3,0.,0.])
rng=np.random.default_rng(18); v=G@rng.normal(size=4)
leak=[]; drift=[]
for k in range(100):
    proposed=.9*v+rng.normal(size=4)
    v=G@proposed
    state=observed+v
    leak.append(np.linalg.norm((I-G)@v))
    drift.append(np.linalg.norm(Oa@(state-observed)))
record('projected_update_max_forbidden_energy',max(leak))
record('projected_update_max_observed_drift',max(drift))
assert max(leak)==0 and max(drift)==0
G0=np.zeros((4,4))
assert np.linalg.matrix_rank(G0)==0
# Empty eligibility is returned as no inferred target, not a confident zero estimate.
recovered=None if np.linalg.matrix_rank(G0)==0 else G0@rng.normal(size=4)
assert recovered is None
record('empty_eligibility_generated_coordinates',0)
# Target-key dependence cannot be diagnosed by source validation alone.
source_conditional=joint_plus; target_conditional=joint_minus
record('target_conditional_reversal_gap',abs(source_conditional[1,1]-target_conditional[1,1])/.5)

## Outputs and limitation
The CSV is a finite witness record, not a posterior-quality or industrial-diagnosis table. Actual learned support projections, source-reference coordinates and industrial LODO evidence remain required. The separate existing coordinate controls are retained.

In [ ]:
out=Path(os.environ.get('TII_BUILD_OUTPUT','outputs/tii_support'))
out.mkdir(parents=True,exist_ok=True)
with (out/'support_witness.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=['witness','value','scope']); w.writeheader(); w.writerows(rows)
print('SUPPORT_POSTERIOR_FINITE_WITNESS_PASS',len(rows))